# 7장 실습: 텍스트로 지식 그래프 구축과 벡터 RAG

실습 가이드 7장 7-1 ~ 7-7의 코드를 순서대로 실행하는 노트북입니다. 설명은 실습 가이드를 함께 봅니다.

| 항목 | 내용 |
|---|---|
| 연결할 DB | 빈 DB (AuraDB 또는 Docker의 빈 컨테이너) |
| .env 설정 | 빈 DB의 NEO4J 네 줄 |

.env를 바꿨다면 노트북 상단의 `Restart`로 커널을 재시작한 뒤 처음부터 실행합니다. 연결 셀의 `connect_graph()`는 .env를 다시 읽지만, 앞 셀에서 만든 객체가 이전 DB를 가리킬 수 있으므로 처음부터 다시 실행합니다.

이 노트북은 DB에 데이터를 씁니다. 다른 데이터가 있는 DB에 연결하면 결과가 섞입니다. 다시 실행할 때는 먼저 DB를 비웁니다(아래 `RESET_DB`).

## 연결 확인

`Neo4jGraph`는 연결할 때 DB 스키마를 읽습니다. 이때 APOC 플러그인을 쓰므로, APOC가 없는 DB에서는 `Could not use APOC procedures` 오류가 납니다(Sandbox, AuraDB, 실습용 Docker 이미지에는 들어 있습니다).

출력된 레이블별 노드 수로 이 차시에 맞는 DB에 연결됐는지 확인합니다.

In [ ]:
from tools.graphrag_tools import connect_graph

# .env의 접속 정보로 연결하고 레이블별 노드 수를 출력한다. expect의 레이블이 없으면 다른 DB에 연결됐다고 알려 준다.
# NEO4J는 접속 정보 묶음이다. 다른 LangChain 객체를 만들 때 **NEO4J로 그대로 넘긴다(실습 가이드 6-2).
# forbid: Recommendations(Movie)나 POLE(Crime)에 연결됐으면 여기서 오류로 멈춘다. 이 노트북은 DB를 지우므로 빈 DB에서만 실행한다.
graph, NEO4J = connect_graph(expect=[], forbid=["Movie", "Crime"])

DB 비우기 (실습 가이드 7-1)

위 출력에 노드가 있으면 이 실습용 DB인지 확인한 뒤 `RESET_DB = True`로 바꿔 실행합니다. 모든 노드와 관계가 지워지고 되돌릴 수 없습니다.

In [ ]:
RESET_DB = False

if RESET_DB:
    graph.query("MATCH (n) DETACH DELETE n")
print("노드 수:", graph.query("MATCH (n) RETURN count(n) AS n")[0]["n"])

## 1. 임베딩 원리: OpenAI API 직접 호출 (실습 가이드 7-3)

LangChain을 쓰기 전에 임베딩이 무엇을 돌려주는지 확인합니다. 같은 뜻의 한국어 문장과 영어 문장을 임베딩해 코사인 유사도를 계산합니다.

In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI()
resp = client.embeddings.create(
    model="text-embedding-3-small",
    input=["마리 퀴리는 노벨상을 두 번 받았다.", "Marie Curie won two Nobel Prizes."],
)
v1, v2 = resp.data[0].embedding, resp.data[1].embedding
print(len(v1))   # 1536

cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
print(round(cos, 3))

## 2. 청크 분할 (실습 가이드 7-4)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample_text = """
Marie Curie, 7 November 1867 - 4 July 1934, was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity.
She was the first woman to win a Nobel Prize, the first person to win a Nobel Prize twice, and the only person to win a Nobel Prize in two scientific fields.
Her husband, Pierre Curie, was a co-winner of her first Nobel Prize, making them the first-ever married couple to win the Nobel Prize and launching the Curie family legacy of five Nobel Prizes.
She was, in 1906, the first woman to become a professor at the University of Paris.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
documents = splitter.create_documents([sample_text])
print(len(documents))
for d in documents:
    print("-", d.page_content)

## 3. 엔티티·관계 추출과 적재 (실습 가이드 7-5)

`LLMGraphTransformer`가 청크마다 gpt-4o를 호출해 노드와 관계를 추출합니다. `allowed_nodes`, `allowed_relationships`로 허용할 이름을 정해 두면 실행마다 이름이 달라지는 문제를 줄일 수 있습니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm = ChatOpenAI(model="gpt-4o", temperature=0)

transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Organization", "Award", "Country", "Field"],
    allowed_relationships=["MARRIED_TO", "WON", "WORKED_AT", "CITIZEN_OF", "RESEARCHED"],
)
graph_documents = transformer.convert_to_graph_documents(documents)

for gd in graph_documents:
    print(gd.nodes)
    print(gd.relationships)

In [ ]:
graph.add_graph_documents(graph_documents, baseEntityLabel=True, include_source=True)

# 적재 결과: 원문 청크(Document)와 엔티티가 MENTIONS로 연결된다.
for row in graph.query("""
MATCH (n) RETURN labels(n) AS labels, count(*) AS n ORDER BY n DESC
"""):
    print(row)

Neo4j Browser에서도 확인합니다.

```cypher
MATCH (d:Document)-[:MENTIONS]->(e)-[r]->(e2)
RETURN d, e, r, e2 LIMIT 50
```

## 4. 벡터 인덱스와 retriever (실습 가이드 7-6)

`Neo4jVector.from_existing_graph`는 `Document` 노드 중 임베딩이 없는 노드를 찾아 임베딩하고 벡터 인덱스까지 만듭니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    **NEO4J,
    index_name="document_index",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
for doc in retriever.invoke("마리 퀴리가 노벨상을 받은 이유는?"):
    print(doc.page_content)
    print("---")

## 5. LCEL RAG 체인 (실습 가이드 7-7)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 문서만 근거로 한국어로 간결하게 답한다. 문서에 없으면 '정보가 부족합니다'라고 답한다."),
    ("human", "문서:\n{context}\n\n질문: {question}"),
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print(rag_chain.invoke("마리 퀴리가 노벨상을 받은 이유는 무엇인가요?"))

## 실습 과제

위의 셀을 모두 실행한 뒤 진행합니다. 과제마다 문제 아래의 빈 셀에 코드를 작성합니다.

### 과제 1. 허용 목록 없이 추출하기

`allowed_nodes`, `allowed_relationships`를 지운 `LLMGraphTransformer`로 같은 청크를 다시 추출하고, 3번 셀의 결과와 레이블·관계 이름을 비교하세요. 이 과제는 DB에 쓰지 않고 Python에서만 비교합니다(과제 2, 3에서 지금 DB를 계속 씁니다).

In [ ]:
# 과제 1: 여기에 작성합니다


### 과제 2. Pierre Curie 문장 추가

Pierre Curie의 경력 문장 4개를 같은 방법(같은 `transformer`)으로 추출해 적재하고, Pierre Curie 노드가 어떤 노드와 연결되는지 확인하세요. 새 청크도 벡터 검색에 쓰이도록 임베딩까지 합니다.

In [ ]:
# 과제 2: 여기에 작성합니다


### 과제 3. k를 1과 3으로 바꿔 비교

같은 질문을 `k=1`과 `k=3`인 retriever로 실행하고, 검색된 청크와 답변을 비교하세요. 질문은 "마리 퀴리가 파리 대학에서 맡은 일은 무엇인가요?"를 씁니다.

In [ ]:
# 과제 3: 여기에 작성합니다
